In [ ]:
%load_ext noworkflow
import noworkflow.now.ipython as nip

In [ ]:
def first(it):
    return next(it)

In [ ]:
def extract_from_result(trial_id, attributes):
    trial = nip.Trial(trial_id)
    
    # SQL query
    sql = first(nip.persistence.query("""
        SELECT value 
        FROM environment_attr
        WHERE name='EXP_NUMBER'
        AND trial_id={}""".format(trial.id)))
    name = 'exp{}.dat'.format(sql['value'])
    
    # Prolog query
    prolog = first(trial.trial_prolog.query("""
        access({0}, _, '{1}', _, _, X, _, _)
    """.format(trial.id, name)))
    hash = prolog['X']

    # Create new file with old content
    content = nip.persistence.get(hash)
    with open('.temp.dat', 'w') as f:
        f.write(content)
    
    # Call external program to extract result
    result = !./extractor .temp.dat $attributes
    
    # Transform result into a dict
    values = result[0].split()
    attrs = attributes.split()
    return {key:values[i] 
            for i, key in enumerate(attrs)}

In [ ]:
extract_from_result(1, "x result")

In [ ]:
extract_from_result(2, "result y")